<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/mini_labs/week_12_xgboost.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BYU CS 473 — XGBoost

In this assignment, you will learn the core ideas behind XGBoost and apply the method to a dataset of your choice.
We’ll connect the math from the textbook to hands-on modeling.

---

## Learning Goals
- Explain the XGBoost objective function and its components.
- Define and use key terms: regularizer, second-order Taylor expansion, leaf weights, gain, and split criterion.
- Apply XGBoost to a dataset, tune hyperparameters, and evaluate results.
- Understand how XGBoost improves upon traditional boosting methods.

## Part 1 — Key Concepts from the Textbook  

Read through the definitions below. For each one, write a **1–2 sentence explanation in your own words**.  

### 1. Regularizer  
Equation (18.47):  
$\Omega(f) = \gamma J + \frac{1}{2} \lambda \sum_{j=1}^J w_j^2$  

**Question:** Why does XGBoost penalize both the **number of leaves** and the **magnitude of leaf weights**?  


number of leaves: This makes it so that the width and depth of the tree stays under constraint---the shorter the tree is, the more pruned it is, so the less it will overfit.


Magnitude of lead weights is penalized because of a few reasons---1) because a high magnitude weight in a tree will be sensitive to perturbations and essentially by unstable, 2) because the high weight will make one tree dominate the others (but our goal in ensembling is to crowdsource/average) and 3) because if all weights are small, we need a combination of many trees in order to get high confidence predictions.


### 2. Second-order Taylor Expansion of the Loss  
Equation (18.49):  
$L_m(F_m) \approx \sum_{i=1}^N \Big[ \ell(y_i, f_{m-1}(x_i)) + g_{im} F_m(x_i) + \tfrac{1}{2} h_{im} F_m(x_i)^2 \Big] + \Omega(F_m)$  

**Question:** How does including the **Hessian term** (curvature) make boosting more accurate compared to using only gradients?  


Hessian takes our loss from being linear (in hyperspace) to being quadratic. Quadratic loss actually much better because the best delta x (for minimum loss) is at the bottom of the parabola (so our optimal step size is based off of this), so we take big steps when needed and small steps when needed.


### 3. Optimal Leaf Weights  
Equation (18.54):  
$w_j^* = - \frac{G_{jm}}{H_{jm} + \lambda}$  

**Question:** What does this formula mean about how leaf weights are chosen?  


we see that this is the negative gradient divided by the hessian plus a regularization coefficient. This


1) points us in the correct direction (negative gradient == down to less loss)


2)  takes small steps when curvature is big (good) and big steps when curvature is small (also good) so our learning rate is dynamic and very efficient


3) lamda stops the learning rate from exploding when the hessian is near zero






### 4. Gain of a Split  
Equation (18.56):  
$\text{gain} = \tfrac{1}{2}\Bigg( \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \Bigg) - \gamma$  

**Question:** Why does XGBoost reject splits with **negative gain**?  


Gain is reduction in loss (with a threshold at gamma), so a negative gain means that the node does not reduce loss enough. It is adding a node without providing at least a gamma-sized benefit, where gamma is a complexity penalty.


## Part 2 — Visualizing Boosting  

### 2.1 Bagging vs Boosting (Recap)  
Describe in words how **bagging** and **boosting** differ in how they:  
- Use data sampling  
- Build models sequentially or in parallel  
- Reduce bias vs variance  



Bagging: we grab a random subset of the data X different times. We are drawing with replacement and therefore this is like building in parallel, because the draws are IID. This techinque reduces variance, because every tree is different and their errors cancel out in the final model.


Boosting: sequential building rather than parallel. We do not choose the next subset randomly, but use the entire dataset every time, weighting the loss function per datapoint more aggressively based on which datapoints had the highest loss LAST iteration. This technique reduces bias.


## Part 3 — Implementing XGBoost on 2 Datasets  

### Step 1 — Look at the example dataset



In [ ]:
# Example: load a dataset
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer

# Load data
X, y = load_breast_cancer(return_X_y=True)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    eta=0.1,        # learning rate
    max_depth=3,    # tree depth
    n_estimators=100,
    reg_lambda=1.0, # L2 regularization
    reg_alpha=0.0   # L1 regularization
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.956140350877193


### Step 2 — Implement XGboost on a dataset of your choice  
- Example locations to find a dataset:  
  - A built-in dataset (e.g. `load_digits`)  
  - A Kaggle dataset  


In [ ]:
# Your code here
import pandas as pd

tit_df = pd.read_csv('train.csv')
# tit_df needs one-hot encoding before we use XGboost well
tit_df = pd.get_dummies(tit_df, columns = ['Sex', 'Embarked'], drop_first=True)
# tit_df['Cabin'] = tit_df['Cabin'].str[0].astype('category')

# tit_df.columns

X_train, X_test, y_train, y_test = train_test_split(
    tit_df[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Sex_male', 'Embarked_Q', 'Embarked_S']],
    tit_df[['Survived']],
    test_size = .25,
    random_state = 42,
    shuffle=True,
    )

X_train['Cabin'] = X_train['Cabin'].str[0].astype('category')

In [ ]:
tit_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    eta=0.1,        # learning rate
    max_depth=3,    # tree depth
    n_estimators=100,
    reg_lambda=1.0, # L2 regularization
    reg_alpha=0.0,   # L1 regularization
    enable_categorical = True
)
tit_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eta=0.1, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None, ...)

Try finding the accuracy on the test set



In [ ]:
X_test['Cabin'] = X_test['Cabin'].str[0].astype('category')

tit_y_predict = tit_model.predict(X_test)
acc = accuracy_score(y_test, tit_y_predict)

print(f"Accuracy: {acc}")



Accuracy: 0.8295964125560538


### Step 3 — Experiment with Hyperparameters on your dataset and the Cancer dataset
- Change `max_depth`, `eta`, or `n_estimators`.  
- Add regularization with `reg_lambda` and `reg_alpha`.  
- **Question:** How do these changes affect performance?  


In [ ]:
new_tit_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    eta=0.2,        # learning rate
    max_depth=3,    # tree depth
    n_estimators=100,
    reg_lambda=1, # L2 regularization
    reg_alpha=0.1,   # L1 regularization
    enable_categorical = True
)
new_tit_model.fit(X_train, y_train)

tit_y_predict = new_tit_model.predict(X_test)
acc = accuracy_score(y_test, tit_y_predict)

print(f"Accuracy: {acc}")



Accuracy: 0.8251121076233184


#For Titanic Dataset:
It is looking like eta doesn't change much. As our max_depth grows past 4, accuracy goes down, and n_estimators doesn't really chagne anything past 30. In fact, we get slightly better results with 30 rather than 50 or 100 estimators in our forest.

We change reg_lambda and find that 1 is about good as it gets, but reg_alpha is best as we are NOT zero but somwhere around .1


# For Cancer Dataset:

In [ ]:

# Load data
X, y = load_breast_cancer(return_X_y=True)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    eta=0.1,        # learning rate
    max_depth=2,    # tree depth
    n_estimators=600,
    reg_lambda=2, # L2 regularization
    reg_alpha=0.1   # L1 regularization
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.9736842105263158


It looks like setting max_depth LOWER, rather than higher is how we are avoiding overfitting and actually reaching a 97.3% test set accuracy.

Cancer dataset is probably something that's very easy to overfit in if we have too-big trees.

eta peaks accuracy around .1, and increasing n_estimators is well worth it even past 100. 600 is peaking the accuracy for estimators.

lambda starts hurting our results when we get very big, which makes sense because our optimizaiton doesn't care about number of leaves but only about weights. lambda hurts as well but mathematically has a slower effect as we increase magnitude. It also isn't squared, so the rate is constant rather than increasing with weight size.

## Part 4 — Reflection  

Answer the following in complete sentences:  
1. What role does the **regularizer** play in preventing overfitting?  
2. How does using the **second-order Taylor expansion** help optimize the trees?  
3. What surprised you most when experimenting with hyperparameters?  
4. Why is XGBoost considered both a **statistical innovation** (Taylor expansion, regularization) and a **computer science innovation** (scalability, out-of-core learning)?  




1)

The regularizer is our Omega, which is a part of the loss function. A higher lambda causes our magnitude of weights to matter much more than our number of weights, and vis versa for a lower lamda (same thing with alpha, just L1 distance not L2). It helps us stay balanced and not allow one specific weight in a tree to outweigh all others (overfit for that specific point), via complexity penalty.

2)

The taylor expansion is just giving us a method to optimize over function space. Because we have so many possible functions, finding the optimal weights like this with newton's method via the hessian (it makes our loss estimation quadratic) is fast. Thus this is faster and more stable than first-order gradient boosting.

3)

I think that I assumed that our regulizers would have more of an effect than they did. Rather, simply the number of trees and what shape we let them become dominated influence over our results.

4)

XGBoost was every bit a mathematical discovery as it was an algorithmic one. It entirely came from deep thinking in mathematical statistics, but is found via computer science's implementation of parallelism/cache-awareness to be incredibly effective, more so than previous boosting algorithms.